# BF Development Analysis with External Loss Development Factors (LDFs)

## Purpose
This notebook implements a comprehensive claims development methodology that:
1. **Count/IBNR development** — from count triangles (same as Part 1)
2. **Severity development** — using external industry LDFs for long-tail perils (BI, subsidence)
3. **Inflation trend** — derived from short-tail perils where development is minimal
4. **Frequency trend** — derived from developed market 2021-2023 data
5. **2025 projection** — applying trends to 2024 ultimate

## Key Change from Part 1
Part 1 applied ONLY count-based development factors (e.g., BI 2024 = 1.82x) to reported incurred.
This assumed severity was fully developed at reported level — **incorrect for long-tail perils**.

This notebook replaces the count-only factors for BI and subsidence with **external age-to-ultimate LDFs**
that capture BOTH count and severity development.

## External LDF Source
**URL:** https://theactuarialclub.com/2019/05/18/loss-development-factors-ldf/

These are cumulative age-to-ultimate factors: Ultimate = Current Incurred × LDF.
Mid-points used where ranges are provided.

## Data Sources
- Developed market: `data/claims.parquet`, `data/proc/proc_*.parquet` (corrected exposure)
- Emerging market: `files_emerging_v1/claims_cleaned.parquet`, `files_emerging_v1/policies_*_cleaned.parquet`

## Important: This notebook does NOT modify any Part 1 or v1 submission-supporting notebooks.
All previous notebooks are preserved unchanged.

In [0]:
import pandas as pd
import numpy as np
import os

BASE = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026'

# ============================================================
# DEVELOPED MARKET
# ============================================================
# Claims (raw — apply same corrections as Part 1)
dev_claims_raw = pd.read_parquet(f'{BASE}/data/claims.parquet')
dev_claims_raw['loss_year'] = pd.to_datetime(dev_claims_raw['loss_date']).dt.year
dev_claims_raw['incurred_net'] = dev_claims_raw['incurred_to_date'] - dev_claims_raw['recoveries']
# Exclude repudiated claims (same as Part 1)
dev_claims = dev_claims_raw[dev_claims_raw['status'] != 'repudiated'].copy()

# Policies (processed — already have corrected exposure + enrichment)
dev_motor = pd.read_parquet(f'{BASE}/data/proc/proc_motor.parquet')
dev_buildings = pd.read_parquet(f'{BASE}/data/proc/proc_buildings.parquet')
dev_contents = pd.read_parquet(f'{BASE}/data/proc/proc_contents.parquet')

dev_exp = {}
for name, df in [('motor', dev_motor), ('buildings', dev_buildings), ('contents', dev_contents)]:
    dev_exp[name] = df.groupby('Year')['exposure'].sum().to_dict()

# ============================================================
# EMERGING MARKET (cleaned files from files_emerging_v1)
# ============================================================
em_claims_raw = pd.read_parquet(f'{BASE}/files_emerging_v1/claims_cleaned.parquet')
em_claims_raw['loss_year'] = pd.to_datetime(em_claims_raw['loss_date']).dt.year
em_claims_raw['incurred_net'] = em_claims_raw['incurred_to_date'] - em_claims_raw['recoveries']
em_claims = em_claims_raw[em_claims_raw['status'] != 'repudiated'].copy()

em_motor = pd.read_parquet(f'{BASE}/files_emerging_v1/policies_motor_cleaned.parquet')
em_buildings = pd.read_parquet(f'{BASE}/files_emerging_v1/policies_buildings_cleaned.parquet')
em_contents = pd.read_parquet(f'{BASE}/files_emerging_v1/policies_contents_cleaned.parquet')

# Add Year column (emerging cleaned files don't have it)
for df in [em_motor, em_buildings, em_contents]:
    df['Year'] = pd.to_datetime(df['period_start']).dt.year

em_exp = {}
for name, df in [('motor', em_motor), ('buildings', em_buildings), ('contents', em_contents)]:
    em_exp[name] = df.groupby('Year')['exposure'].sum().to_dict()

print(f"Developed claims: {len(dev_claims):,} active, years {dev_claims['loss_year'].min()}-{dev_claims['loss_year'].max()}")
print(f"Emerging claims: {len(em_claims):,} active, years {em_claims['loss_year'].min()}-{em_claims['loss_year'].max()}")
print(f"Developed exposure (motor 2024): {dev_exp['motor'][2024]:,.0f}")
print(f"Emerging exposure (motor 2024): {em_exp['motor'][2024]:,.0f}")

In [0]:
# ============================================================
# EXTERNAL LOSS DEVELOPMENT FACTORS
# Source: https://theactuarialclub.com/2019/05/18/loss-development-factors-ldf/
# These are cumulative age-to-ultimate factors (Incurred LDFs).
# Ultimate = Current Incurred × LDF
# Mid-points used where ranges are provided.
# ============================================================

# Raw ranges from source
RAW_LDFS = {
    'bodily_injury': {
        12: (1.85, 2.40),   # 12 months: highly volatile
        24: (1.40, 1.65),
        36: (1.20, 1.35),
        48: (1.10, 1.18),
        60: (1.05, 1.08),
        'tail': (1.02, 1.05),  # Tail factor (60+ months)
    },
    'subsidence': {
        12: (1.50, 1.80),   # 12 months: heavily dependent on surge status
        24: (1.15, 1.25),
        36: (1.05, 1.10),
        48: (1.01, 1.03),
        60: (1.00, 1.00),   # Typically fully developed at 60 months
        'tail': (1.00, 1.00),
    }
}

# Compute mid-points
LDF_MIDPOINTS = {}
for peril, factors in RAW_LDFS.items():
    LDF_MIDPOINTS[peril] = {age: (r[0] + r[1]) / 2 for age, r in factors.items()}

print('=== EXTERNAL LDFs (MID-POINTS) ===')
print(f"{'Age (mo)':>10} {'BI':>10} {'Subsidence':>12}")
for age in [12, 24, 36, 48, 60]:
    print(f"{age:>10} {LDF_MIDPOINTS['bodily_injury'][age]:>10.3f} {LDF_MIDPOINTS['subsidence'][age]:>12.3f}")
print(f"{'Tail':>10} {LDF_MIDPOINTS['bodily_injury']['tail']:>10.3f} {LDF_MIDPOINTS['subsidence']['tail']:>12.3f}")

# ============================================================
# MAP LOSS YEARS TO DEVELOPMENT AGES
# Data snapshot: end of December 2024
# Average loss date: ~mid-year (July 1)
# ============================================================
SNAPSHOT_DATE = pd.Timestamp('2024-12-31')

loss_years = [2021, 2022, 2023, 2024]
dev_ages = {}
for yr in loss_years:
    avg_loss_date = pd.Timestamp(f'{yr}-07-01')
    dev_ages[yr] = (SNAPSHOT_DATE - avg_loss_date).days / 30.44

# Interpolate LDFs for each loss year
def interpolate_ldf(peril, age_months):
    ldf_table = LDF_MIDPOINTS[peril]
    standard_ages = [12, 24, 36, 48, 60]
    if age_months <= 12:
        return ldf_table[12]  # Conservative: use 12-month for <12 months
    elif age_months >= 60:
        return ldf_table[60] * ldf_table['tail']  # Apply tail on top of 60-month
    else:
        for i in range(len(standard_ages) - 1):
            if standard_ages[i] <= age_months <= standard_ages[i+1]:
                lower, upper = standard_ages[i], standard_ages[i+1]
                frac = (age_months - lower) / (upper - lower)
                return ldf_table[lower] + frac * (ldf_table[upper] - ldf_table[lower])
    return ldf_table[60]

YEAR_LDFS = {}
print('\n=== INTERPOLATED LDFs BY LOSS YEAR ===')
print(f"{'Loss Year':>10} {'Dev Age (mo)':>14} {'BI LDF':>10} {'Subs LDF':>10}")
for yr in loss_years:
    age = dev_ages[yr]
    bi_ldf = interpolate_ldf('bodily_injury', age)
    sub_ldf = interpolate_ldf('subsidence', age)
    YEAR_LDFS[yr] = {'bodily_injury': bi_ldf, 'subsidence': sub_ldf}
    print(f"{yr:>10} {age:>13.1f} {bi_ldf:>10.3f} {sub_ldf:>10.3f}")

print('\nNote: Even 2021 gets further development (BI: 1.207x, Subsidence: 1.047x)')
print('This addresses the key gap in Part 1, which assumed 2021 was fully developed.')

In [0]:
# ============================================================
# INFLATION TREND (from short-tail perils, closed claims)
# Short-tail perils are fully developed by 2021-2022, so year-on-year
# severity change is pure inflation (no development artifact).
# ============================================================

dev_closed = dev_claims[dev_claims['status'] == 'closed'].copy()

PERIL_TAIL = {
    'bodily_injury': 'long', 'subsidence': 'long',
    'third_party_prop': 'medium', 'fire': 'medium',
    'own_damage': 'short', 'theft': 'short', 'glass_hail': 'short',
    'escape_of_water': 'short', 'flood': 'short', 'storm': 'short', 'accidental_damage': 'short',
}

print('=== INFLATION TREND (short-tail perils, closed claims, 2021-2023) ===')
inflation_by_line = {}
for line in ['motor', 'buildings', 'contents']:
    rates = []
    for peril in sorted(dev_claims[dev_claims['line'] == line]['peril'].unique()):
        if PERIL_TAIL.get(peril) != 'short':
            continue
        closed_data = dev_closed[(dev_closed['line'] == line) & (dev_closed['peril'] == peril)]
        by_year = closed_data.groupby('loss_year')['incurred_net'].mean()
        if 2021 in by_year.index and 2023 in by_year.index and by_year[2021] > 0:
            rate = (by_year[2023] / by_year[2021]) ** (1/2) - 1
            rates.append(rate)
            print(f'  {line}/{peril}: {rate*100:.1f}%/yr')
    if rates:
        inflation_by_line[line] = np.mean(rates)
        print(f'  --> {line.upper()} average: {inflation_by_line[line]*100:.1f}%/yr\n')

# ============================================================
# FREQUENCY TREND (2021-2023, avoiding 2024 IBNR)
# ============================================================
print('\n=== FREQUENCY TREND (2021-2023) ===')
freq_trend_by_peril = {}
for line in ['motor', 'buildings', 'contents']:
    for peril in sorted(dev_claims[dev_claims['line'] == line]['peril'].unique()):
        counts = dev_claims[(dev_claims['line'] == line) & (dev_claims['peril'] == peril)].groupby('loss_year')['claim_id'].count()
        freqs = {}
        for yr in [2021, 2022, 2023]:
            exp = dev_exp.get(line, {}).get(yr, 0)
            if yr in counts.index and exp > 0:
                freqs[yr] = counts[yr] / exp
        if 2021 in freqs and 2023 in freqs:
            trend = (freqs[2023] / freqs[2021]) ** (1/2) - 1
        else:
            trend = 0.0
        freq_trend_by_peril[(line, peril)] = trend
        print(f'  {line}/{peril}: {trend*100:+.1f}%/yr')

In [0]:
# ============================================================
# DEVELOPMENT FACTOR FUNCTION
# For BI and subsidence: use external LDFs (captures count + severity dev)
# For all other perils: use Part 1 count-based factors (severity already developed)
# ============================================================

# Part 1 count-based factors (for non-BI, non-subsidence perils)
P1_COUNT_FACTORS = {
    2021: {'tpp': 1.00, 'short_motor': 1.00, 'buildings_other': 1.00, 'contents': 1.00},
    2022: {'tpp': 1.01, 'short_motor': 1.01, 'buildings_other': 1.02, 'contents': 1.01},
    2023: {'tpp': 1.01, 'short_motor': 1.01, 'buildings_other': 1.02, 'contents': 1.01},
    2024: {'tpp': 1.08, 'short_motor': 1.08, 'buildings_other': 1.08, 'contents': 1.04},
}

def get_dev_factor(peril, line, loss_year):
    peril_lower = str(peril).lower()
    # External LDF for BI and subsidence (replaces Part 1 count-only factors)
    if peril_lower == 'bodily_injury':
        return YEAR_LDFS.get(loss_year, {}).get('bodily_injury', 1.0)
    elif peril_lower == 'subsidence':
        return YEAR_LDFS.get(loss_year, {}).get('subsidence', 1.0)
    else:
        # Part 1 count-based factor for short/medium-tail perils
        if line == 'motor':
            tail = 'tpp' if 'third_party' in peril_lower else 'short_motor'
        elif line == 'buildings':
            tail = 'buildings_other'
        else:
            tail = 'contents'
        return P1_COUNT_FACTORS.get(loss_year, {}).get(tail, 1.0)

# Apply to both markets
for market_name, claims_df in [('DEVELOPED', dev_claims), ('EMERGING', em_claims)]:
    claims_df['dev_factor'] = claims_df.apply(
        lambda r: get_dev_factor(r['peril'], r['line'], r['loss_year']), axis=1
    )
    claims_df['ultimate_incurred'] = claims_df['incurred_net'] * claims_df['dev_factor']

# Aggregate
dev_summary = dev_claims.groupby(['line', 'peril', 'loss_year', 'dev_factor']).agg(
    claim_count=('claim_id', 'count'),
    reported_incurred=('incurred_net', 'sum'),
    ultimate_incurred=('ultimate_incurred', 'sum'),
).reset_index()
dev_summary['exposure'] = dev_summary.apply(lambda r: dev_exp.get(r['line'], {}).get(r['loss_year'], 0), axis=1)
dev_summary['reported_lr'] = dev_summary.apply(lambda r: r['reported_incurred']/r['exposure'] if r['exposure'] > 0 else 0, axis=1)
dev_summary['ultimate_lr'] = dev_summary.apply(lambda r: r['ultimate_incurred']/r['exposure'] if r['exposure'] > 0 else 0, axis=1)
dev_summary['avg_sev_ult'] = dev_summary['ultimate_incurred'] / dev_summary['claim_count']

em_summary = em_claims.groupby(['line', 'peril', 'loss_year', 'dev_factor']).agg(
    claim_count=('claim_id', 'count'),
    reported_incurred=('incurred_net', 'sum'),
    ultimate_incurred=('ultimate_incurred', 'sum'),
).reset_index()
em_summary['exposure'] = em_summary.apply(lambda r: em_exp.get(r['line'], {}).get(r['loss_year'], 0), axis=1)
em_summary['reported_lr'] = em_summary.apply(lambda r: r['reported_incurred']/r['exposure'] if r['exposure'] > 0 else 0, axis=1)
em_summary['ultimate_lr'] = em_summary.apply(lambda r: r['ultimate_incurred']/r['exposure'] if r['exposure'] > 0 else 0, axis=1)
em_summary['avg_sev_ult'] = em_summary['ultimate_incurred'] / em_summary['claim_count']

# Print summary
for market_name, summary_df, exp_dict in [('DEVELOPED', dev_summary, dev_exp), ('EMERGING', em_summary, em_exp)]:
    print(f'\n=== {market_name} MARKET: TOTAL BY YEAR ===')
    print(f"{'Year':>4} {'Rep_LR':>10} {'Ult_LR':>10} {'Uplift%':>8}")
    for yr in sorted(summary_df[summary_df['exposure'] > 0]['loss_year'].unique()):
        yr_data = summary_df[(summary_df['loss_year'] == yr) & (summary_df['exposure'] > 0)]
        if len(yr_data) == 0:
            continue
        total_exp = yr_data['exposure'].sum()
        rep_total = (yr_data['reported_lr'] * yr_data['exposure']).sum() / total_exp
        ult_total = (yr_data['ultimate_lr'] * yr_data['exposure']).sum() / total_exp
        uplift = (ult_total / rep_total - 1) * 100 if rep_total > 0 else 0
        print(f"{yr:>4} {rep_total:>10.1f} {ult_total:>10.1f} {uplift:>7.1f}%")

In [0]:
# ============================================================
# 2025 PROJECTION
# 2025_ultimate = 2024_ultimate_LR × (1 + frequency_trend) × (1 + inflation)
# ============================================================

for market_name, summary_df, exp_dict in [('DEVELOPED', dev_summary, dev_exp), ('EMERGING', em_summary, em_exp)]:
    print(f'\n=== {market_name} 2025 PROJECTION ===')
    print(f"{'Line':<10} {'Peril':<22} {'2024_Ult':>10} {'Freq_tr':>8} {'Infl':>7} {'2025_Ult':>10} {'2025/24':>8}")
    total_2024, total_2025, total_exp = 0, 0, 0
    for line in ['motor', 'buildings', 'contents']:
        infl = inflation_by_line.get(line, 0.05)
        exp_2024 = exp_dict.get(line, {}).get(2024, 0)
        total_exp += exp_2024
        for peril in sorted(summary_df[summary_df['line'] == line]['peril'].unique()):
            d = summary_df[(summary_df['line'] == line) & (summary_df['peril'] == peril) & (summary_df['loss_year'] == 2024) & (summary_df['exposure'] > 0)]
            if len(d) == 0:
                continue
            lr_2024 = d['ultimate_lr'].values[0]
            freq_tr = freq_trend_by_peril.get((line, peril), 0)
            lr_2025 = lr_2024 * (1 + freq_tr) * (1 + infl)
            total_2024 += lr_2024 * exp_2024
            total_2025 += lr_2025 * exp_2024
            print(f"{line:<10} {peril:<22} {lr_2024:>10.1f} {freq_tr*100:>7.1f}% {infl*100:>6.1f}% {lr_2025:>10.1f} {lr_2025/lr_2024:>7.2f}x")
    print(f"{'TOTAL':>33} {total_2024/total_exp:>10.1f} {'':>8} {'':>7} {total_2025/total_exp:>10.1f} {total_2025/total_2024:>7.2f}x")

# ============================================================
# PART 1 vs NEW COMPARISON
# ============================================================
print('\n\n=== PART 1 vs NEW APPROACH (DEVELOPED MARKET 2024) ===')
print(f"{'Metric':<40} {'Part 1':>12} {'New':>12} {'New/P1':>8}")
print('-' * 75)

p1_2024, new_2024, exp_total = 0, 0, 0
for line in ['motor', 'buildings', 'contents']:
    exp = dev_exp.get(line, {}).get(2024, 0)
    exp_total += exp
    for peril in sorted(dev_claims[dev_claims['line'] == line]['peril'].unique()):
        yr_data = dev_claims[(dev_claims['line'] == line) & (dev_claims['peril'] == peril) & (dev_claims['loss_year'] == 2024)]
        if len(yr_data) == 0:
            continue
        rep_inc = yr_data['incurred_net'].sum()
        new_factor = get_dev_factor(peril, line, 2024)
        # Part 1 original factors (count-only)
        if peril == 'bodily_injury':
            p1_factor = 1.82
        elif peril == 'subsidence':
            p1_factor = 1.47
        elif line == 'motor' and 'third_party' in peril:
            p1_factor = 1.08
        elif line == 'motor':
            p1_factor = 1.08
        elif line == 'buildings':
            p1_factor = 1.08
        else:
            p1_factor = 1.04
        p1_2024 += rep_inc * p1_factor
        new_2024 += rep_inc * new_factor

print(f"{'2024 Ultimate LR':<40} {p1_2024/exp_total:>12.1f} {new_2024/exp_total:>12.1f} {new_2024/p1_2024:>7.2f}x")
print(f"{'2021 gets further development':<40} {'No':>12} {'Yes':>12}")
print(f"{'BI factor source':<40} {'Count triangle':>12} {'External LDF':>12}")
print(f"{'External source':<40} {'N/A':>12} {'theactuarialclub.com':>12}")

In [0]:
# ============================================================
# STEP 7: UPDATE EMERGING PROC FILES WITH NEW ULTIMATE_CLAIMS
# Recalculate ultimate_claims per policy using external LDFs for BI/subsidence
# and Part 1 count factors for short-tail perils.
# ============================================================

DATA_DIR = f'{BASE}/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load emerging claims (already done in Step 1, but recompute to be safe)
em_claims_update = em_claims.copy()

# Compute old (Part 1) and new ultimate per claim
P1_FACTORS = {
    2021: {'bi': 1.00, 'tpp': 1.00, 'short_motor': 1.00, 'subsidence': 1.00, 'buildings_other': 1.00, 'contents': 1.00},
    2022: {'bi': 1.01, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.05, 'buildings_other': 1.02, 'contents': 1.01},
    2023: {'bi': 1.06, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.15, 'buildings_other': 1.02, 'contents': 1.01},
    2024: {'bi': 1.82, 'tpp': 1.08, 'short_motor': 1.08, 'subsidence': 1.47, 'buildings_other': 1.08, 'contents': 1.04},
}

def get_p1_factor(peril, line, yr):
    p = str(peril).lower()
    if p == 'bodily_injury': tail = 'bi'
    elif p == 'subsidence': tail = 'subsidence'
    elif line == 'motor' and 'third_party' in p: tail = 'tpp'
    elif line == 'motor': tail = 'short_motor'
    elif line == 'buildings': tail = 'buildings_other'
    else: tail = 'contents'
    return P1_FACTORS.get(yr, {}).get(tail, 1.0)

em_claims_update['p1_factor'] = em_claims_update.apply(lambda r: get_p1_factor(r['peril'], r['line'], r['loss_year']), axis=1)
em_claims_update['new_factor'] = em_claims_update.apply(lambda r: get_dev_factor(r['peril'], r['line'], r['loss_year']), axis=1)

# For each claim, compute the ratio of new to old ultimate
# This ratio scales the claim's contribution to ultimate_claims
em_claims_update['factor_ratio'] = em_claims_update['new_factor'] / em_claims_update['p1_factor']

# Aggregate by policy_id: weighted average of factor ratios (weighted by incurred_net)
# This gives us the scaling factor for each policy's ultimate_claims
policy_scaling = em_claims_update.groupby(['line', 'policy_id']).apply(
    lambda g: np.average(g['factor_ratio'], weights=g['incurred_net']) if g['incurred_net'].sum() > 0 else 1.0
).reset_index(name='scaling_factor')

print('=== POLICY-LEVEL SCALING FACTORS ===')
for line in ['motor', 'buildings', 'contents']:
    line_scaling = policy_scaling[policy_scaling['line'] == line]
    print(f'{line}: {len(line_scaling)} policies with claims, mean scale={line_scaling["scaling_factor"].mean():.4f}, range=[{line_scaling["scaling_factor"].min():.4f}, {line_scaling["scaling_factor"].max():.4f}]')

# Update proc files
train_updated = {}
old_means = {}
new_means = {}
for line in ['motor', 'buildings', 'contents']:
    train_updated[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    old_means[line] = train_updated[line]['ultimate_claims'].mean()
    
    # Merge scaling factors
    line_scaling = policy_scaling[policy_scaling['line'] == line][['policy_id', 'scaling_factor']]
    train_updated[line] = train_updated[line].merge(line_scaling, on='policy_id', how='left')
    train_updated[line]['scaling_factor'] = train_updated[line]['scaling_factor'].fillna(1.0)  # No claims = no change
    
    # Scale ultimate_claims
    train_updated[line]['ultimate_claims'] = train_updated[line]['ultimate_claims'] * train_updated[line]['scaling_factor']
    train_updated[line].drop('scaling_factor', axis=1, inplace=True)
    
    new_means[line] = train_updated[line]['ultimate_claims'].mean()
    print(f'{line}: old mean={old_means[line]:.4f}, new mean={new_means[line]:.4f}, ratio={new_means[line]/old_means[line]:.4f}')

# Save updated proc files
V3_DIR = f'{BASE}/files_emerging_v2/'
os.makedirs(V3_DIR, exist_ok=True)
for line in ['motor', 'buildings', 'contents']:
    save_path = os.path.join(V3_DIR, f'proc_{line}_v3.parquet')
    train_updated[line].to_parquet(save_path, index=False)
    print(f'Saved: {save_path}')

In [0]:
# ============================================================
# STEP 8: RE-RUN XGBOOST + GLM WITH UPDATED TARGETS (v3 proc files)
# Compare D² with v2 baseline to see if external LDFs improve model fit
# ============================================================

%pip install xgboost statsmodels -q

import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import pickle

# Feature sets (same as v2)
FEATURE_SETS = {
    'motor': ['exposure', 'Year', 'driver_age', 'ncb_years', 'vehicle_value_group',
              'vehicle_age', 'cover_type', 'fuel_type', 'channel',
              'missed_payments_12m', 'prior_claims_5y', 'urban_density', 'renewal_count'],
    'buildings': ['exposure', 'Year', 'sum_insured', 'construction_type', 'voluntary_excess',
                  'missed_payments_12m', 'prior_claims_5y', 'flood_zone', 'elevation_m',
                  'year_built', 'occupancy', 'channel', 'new_business'],
    'contents': ['exposure', 'Year', 'sum_insured', 'security_level', 'voluntary_excess',
                 'missed_payments_12m', 'prior_claims_5y', 'flood_zone', 'elevation_m',
                 'urban_density', 'channel', 'accidental_damage_cover', 'tenure'],
}

# Tweedie deviance/D² functions
def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

# Load updated (v3) proc files
V3_DIR = f'{BASE}/files_emerging_v2/'
train_v3 = {}
for line in ['motor', 'buildings', 'contents']:
    train_v3[line] = pd.read_parquet(os.path.join(V3_DIR, f'proc_{line}_v3.parquet'))

# Prepare test data (same as v2 notebook)
AGE_BAND_MAP = {'17-24': 20.5, '25-34': 29.5, '35-49': 42.0, '50-64': 57.0, '65+': 70.0}
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    for col in df.columns:
        if col.endswith('_age_band') or col == 'age_band':
            new_col = col.replace('_age_band', '_age')
            df[new_col] = df[col].map(AGE_BAND_MAP)
    if 'compulsory_excess' in df.columns and df['compulsory_excess'].nunique() <= 1:
        df.drop('compulsory_excess', axis=1, inplace=True)

customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

cust_lens = customers['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    max_len = cust_lens.index.max()
    customers['customer_id'] = customers['customer_id'].str.zfill(max_len)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    if 'customer_id' in df.columns:
        lens = df['customer_id'].str.len().value_counts()
        if lens.nunique() > 1:
            max_len = lens.index.max()
            df['customer_id'] = df['customer_id'].str.zfill(max_len)

customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df
    for f in FEATURE_SETS[line]:
        if f not in df.columns:
            print(f'WARNING: {line} test missing feature {f}')

# New trend factors (inflation × frequency, computed in Steps 3-5)
NEW_TREND = {'motor': 1.048, 'buildings': 1.090, 'contents': 1.049}  # From Step 5 output (freq trend × inflation)

print(f'New trend factors: {NEW_TREND}')
print(f'Old trend factors: motor=1.04, buildings=1.096, contents=1.049')

# ============================================================
# XGBoost CV (same params as v2)
# ============================================================
N_FOLDS = 5
xgb_params = {
    'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5,
    'max_depth': 2, 'n_estimators': 50, 'min_child_weight': 50,
    'reg_alpha': 2.0, 'reg_lambda': 5.0, 'learning_rate': 0.1,
    'random_state': 42, 'n_jobs': -1,
}

xgb_cv_v3 = {}
xgb_oof_v3 = {}
xgb_test_v3 = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n=== {line.upper()} XGBoost v3 (external LDF targets) ===')
    df_train = train_v3[line]
    df_test = test[line]
    features = [f for f in FEATURE_SETS[line] if f in df_train.columns and f in df_test.columns]
    y_all = df_train['ultimate_claims'].values
    X_all = df_train[features].copy()
    X_test = df_test[features].copy()

    # Encode categoricals
    for col in features:
        if X_all[col].dtype == 'object' or str(X_all[col].dtype) == 'category':
            combined = pd.concat([X_all[col], X_test[col]], axis=0).astype(str)
            codes = combined.astype('category').cat.codes
            X_all[col] = codes.iloc[:len(X_all)].values
            X_test[col] = codes.iloc[len(X_all):].values

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    oof_pred = np.zeros(len(df_train))
    test_pred = np.zeros(len(df_test))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(df_train)):
        X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
        y_tr, y_val = y_all[tr_idx], y_all[val_idx]

        model = xgb.XGBRegressor(**xgb_params)
        model.fit(X_tr, y_tr)
        val_pred = model.predict(X_val)
        oof_pred[val_idx] = val_pred
        test_pred += model.predict(X_test) / N_FOLDS

        d2 = tweedie_d2(y_val, val_pred)
        fold_scores.append(d2)
        print(f'  Fold {fold+1}: D2 = {d2:.4f}')

    cv_d2 = np.mean(fold_scores)
    oof_d2 = tweedie_d2(y_all, oof_pred)
    xgb_cv_v3[line] = cv_d2
    xgb_oof_v3[line] = oof_pred
    xgb_test_v3[line] = test_pred
    print(f'  CV mean D2: {cv_d2:.4f} | OOF D2: {oof_d2:.4f}')
    print(f'  (v2 was: {0.0477 if line=="motor" else 0.0603 if line=="buildings" else 0.0725:.4f})')

print(f'\nXGBoost v3 average D2: {np.mean(list(xgb_cv_v3.values())):.4f} (v2 was 0.0602)')

# ============================================================
# GLM CV (Tweedie, same features as v2)
# ============================================================
glm_cv_v3 = {}
glm_oof_v3 = {}
glm_test_v3 = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n=== {line.upper()} GLM v3 ===')
    df_train = train_v3[line]
    df_test = test[line]
    features = [f for f in FEATURE_SETS[line] if f in df_train.columns and f in df_test.columns]
    y_all = df_train['ultimate_claims'].values
    exp_all = df_train['exposure'].values
    exp_test = df_test['exposure'].values

    # Identify categorical vs numerical features ONCE
    cat_cols = [c for c in features if df_train[c].dtype == 'object' or str(df_train[c].dtype) == 'category' or df_train[c].dtype == 'bool']
    num_cols = [c for c in features if c not in cat_cols]

    # Fill NaN in numerical features (from enrichment merge gaps)
    for col in num_cols:
        med = df_train[col].median()
        df_train[col] = df_train[col].fillna(med)
        df_test[col] = df_test[col].fillna(med)

    # Fill NaN in categorical features with 'unknown'
    for col in cat_cols:
        df_train[col] = df_train[col].astype(str).fillna('unknown')
        df_test[col] = df_test[col].astype(str).fillna('unknown')

    # Make a pristine copy of test features for encoding per fold
    X_test_template = df_test[features].copy()

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    oof_pred = np.zeros(len(df_train))
    test_pred = np.zeros(len(df_test))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(df_train)):
        X_tr = df_train.iloc[tr_idx][features].copy()
        X_val = df_train.iloc[val_idx][features].copy()
        X_test_fold = X_test_template.copy()
        y_tr, y_val = y_all[tr_idx], y_all[val_idx]
        e_tr, e_val = exp_all[tr_idx], exp_all[val_idx]

        # Target-guided encoding for categoricals (from training fold only)
        for col in cat_cols:
            prior = y_tr.mean()
            grp = pd.DataFrame({'cat': X_tr[col].astype(str), 'target': y_tr})
            means = grp.groupby('cat')['target'].mean()
            counts = grp.groupby('cat')['target'].count()
            smoothed = (means * counts + prior * 50) / (counts + 50)
            X_tr[col] = X_tr[col].astype(str).map(smoothed).fillna(prior)
            X_val[col] = X_val[col].astype(str).map(smoothed).fillna(prior)
            X_test_fold[col] = X_test_fold[col].astype(str).map(smoothed).fillna(prior)

        X_tr_glm = sm.add_constant(X_tr.astype(float), has_constant='add')
        X_val_glm = sm.add_constant(X_val.astype(float), has_constant='add')
        X_test_glm = sm.add_constant(X_test_fold.astype(float), has_constant='add')

        try:
            model = sm.GLM(y_tr, X_tr_glm, family=sm.families.Tweedie(var_power=1.5),
                          offset=np.log(np.maximum(e_tr, 0.01)))
            result = model.fit(maxiter=100, disp=0)
            val_pred = result.predict(X_val_glm, offset=np.log(np.maximum(e_val, 0.01)))
            test_pred += result.predict(X_test_glm, offset=np.log(np.maximum(exp_test, 0.01))) / N_FOLDS
        except Exception as e:
            print(f'  Fold {fold+1} failed: {e}')
            val_pred = np.full(len(val_idx), y_tr.mean())
            test_pred += np.full(len(df_test), y_tr.mean()) / N_FOLDS

        oof_pred[val_idx] = val_pred
        d2 = tweedie_d2(y_val, val_pred)
        fold_scores.append(d2)
        print(f'  Fold {fold+1}: D2 = {d2:.4f}')

    cv_d2 = np.mean(fold_scores)
    oof_d2 = tweedie_d2(y_all, oof_pred)
    glm_cv_v3[line] = cv_d2
    glm_oof_v3[line] = oof_pred
    glm_test_v3[line] = test_pred
    print(f'  CV mean D2: {cv_d2:.4f} | OOF D2: {oof_d2:.4f}')
    print(f'  (v2 was: {0.0432 if line=="motor" else 0.0388 if line=="buildings" else 0.0842:.4f})')

print(f'\nGLM v3 average D2: {np.mean(list(glm_cv_v3.values())):.4f} (v2 was 0.0554)')

# ============================================================
# BLEND + SUBMISSION
# ============================================================
print('\n=== BLEND v3 (XGB v3 + GLM v3) ===')
best_weights_v3 = {}
blend_d2_v3 = {}

for line in ['motor', 'buildings', 'contents']:
    y_all = train_v3[line]['ultimate_claims'].values
    best_d2 = -999
    best_w = (0.5, 0.5)
    for wx in np.arange(0, 1.01, 0.05):
        wg = 1.0 - wx
        blend = wx * xgb_oof_v3[line] + wg * glm_oof_v3[line]
        d2 = tweedie_d2(y_all, blend)
        if d2 > best_d2:
            best_d2 = d2
            best_w = (wx, wg)
    blend_d2_v3[line] = best_d2
    best_weights_v3[line] = best_w
    print(f'{line}: XGB={best_w[0]:.2f}, GLM={best_w[1]:.2f} -> D2={best_d2:.4f}')

print(f'\nBlend v3 average D2: {np.mean(list(blend_d2_v3.values())):.4f}')
print(f'Blend v2 average D2 was: 0.0672')

# Generate calibrated submission
submission_v3 = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    wx, wg = best_weights_v3[line]
    xgb_p = np.clip(xgb_test_v3[line], 0.01, 50000) * NEW_TREND[line]
    glm_p = np.clip(np.nan_to_num(glm_test_v3[line], posinf=50000, neginf=0.01), 0.01, 50000) * NEW_TREND[line]
    blend_pred = np.clip(wx * xgb_p + wg * glm_p, 0.01, 50000)
    cal_target = train_v3[line]['ultimate_claims'].mean() * NEW_TREND[line]
    cal_factor = cal_target / blend_pred.mean()
    blend_pred = np.clip(blend_pred * cal_factor, 0.01, 50000)
    print(f'{line}: blend mean={blend_pred.mean():.4f} (target={cal_target:.4f})')
    pred_map = dict(zip(test_df['row_id'], blend_pred))
    mask = submission_v3['row_id'].isin(pred_map)
    submission_v3.loc[mask, 'prediction'] = submission_v3.loc[mask, 'row_id'].map(pred_map)

submission_v3['prediction'] = submission_v3['prediction'].clip(lower=0.01, upper=50000)
submission_v3['prediction'] = submission_v3['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)

SUB_DIR = f'{BASE}/submissions_emerging_v2/'
os.makedirs(SUB_DIR, exist_ok=True)
sub_path_v3 = os.path.join(SUB_DIR, 'blend_v3_external_LDFs.csv')
submission_v3.to_csv(sub_path_v3, index=False)
print(f'\nSaved: {sub_path_v3} ({len(submission_v3)} rows, mean={submission_v3["prediction"].mean():.4f})')
print(f'(v2 submission mean was: 299.7552)')

# Save v3 results for reuse
v3_results = {
    'xgb_cv_scores': xgb_cv_v3,
    'glm_cv_scores': glm_cv_v3,
    'blend_d2': blend_d2_v3,
    'best_weights': best_weights_v3,
    'trend_factors': NEW_TREND,
}
with open(os.path.join(V3_DIR, 'v3_results.pkl'), 'wb') as f:
    pickle.dump(v3_results, f)
print(f'Saved: {V3_DIR}v3_results.pkl')

In [0]:
%pip install openpyxl -q

# ============================================================
# SAVE EXCEL WORKBOOK
# ============================================================

EXCEL_DIR = f'{BASE}/Excel files_emerging_v2/'
os.makedirs(EXCEL_DIR, exist_ok=True)
excel_path = os.path.join(EXCEL_DIR, 'BF_Development_with_External_LDFs.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Sheet 1: Methodology
    methodology = pd.DataFrame({
        'Item': [
            'Count/IBNR Development', 'Severity Development (BI)', 'Severity Development (Subsidence)',
            'Short-tail perils', 'Inflation trend', 'Frequency trend',
            '2021 further development', 'External LDF source', '2025 projection method',
            'Key change from Part 1',
        ],
        'Approach': [
            'Part 1 count triangles (unchanged for short-tail)',
            'External age-to-ultimate LDFs (replaces Part 1 count-only factor)',
            'External age-to-ultimate LDFs (replaces Part 1 count-only factor)',
            'Part 1 count-based factors (no severity development needed)',
            'Derived from short-tail closed-claim severity 2021-2023',
            'Derived from developed market frequency 2021-2023 (avoids 2024 IBNR)',
            'Yes - BI 1.207x, Subsidence 1.047x at ~42 months',
            'https://theactuarialclub.com/2019/05/18/loss-development-factors-ldf/',
            '2024 ultimate × (1+freq trend) × (1+inflation)',
            'Part 1 applied count factor only; now BI/subsidence use external LDFs capturing both count+severity',
        ],
        'Details': [
            'e.g. 2024 BI: count factor was 1.82 (count triangle only)',
            '2024 BI: 2.125 (12-month mid-point from external source)',
            '2024 Subsidence: 1.650 (12-month mid-point from external source)',
            'Own damage, theft, glass, EOW, flood, storm, AD: ~1.0x severity dev',
            f'Motor: {inflation_by_line.get("motor",0)*100:.1f}%/yr, Buildings: {inflation_by_line.get("buildings",0)*100:.1f}%/yr, Contents: {inflation_by_line.get("contents",0)*100:.1f}%/yr',
            'Varies by peril: -5% to +12%/yr',
            'Part 1 assumed 2021 fully developed (factor 1.00); now correctly developed',
            'Mid-points used where ranges provided',
            'Applies to both developed and emerging markets',
            'Net effect: 2024 dev LR ~431 vs Part 1 ~415 (4% increase)',
        ]
    })
    methodology.to_excel(writer, sheet_name='Methodology', index=False)

    # Sheet 2: External LDFs
    ldf_table = pd.DataFrame({
        'Age_Months': [12, 24, 36, 48, 60, 'Tail'],
        'BI_Range': ['1.85-2.40', '1.40-1.65', '1.20-1.35', '1.10-1.18', '1.05-1.08', '1.02-1.05'],
        'BI_Midpoint': [2.125, 1.525, 1.275, 1.14, 1.065, 1.035],
        'Subsidence_Range': ['1.50-1.80', '1.15-1.25', '1.05-1.10', '1.01-1.03', '1.00', '1.00'],
        'Subsidence_Midpoint': [1.65, 1.20, 1.075, 1.02, 1.00, 1.00],
    })
    ldf_table.to_excel(writer, sheet_name='External_LDFs', index=False)

    # Sheet 3: LDFs by loss year
    year_ldf_df = pd.DataFrame([
        {'Loss_Year': yr, 'Dev_Age_Months': dev_ages[yr],
         'BI_LDF': YEAR_LDFS[yr]['bodily_injury'], 'Subsidence_LDF': YEAR_LDFS[yr]['subsidence']}
        for yr in loss_years
    ])
    year_ldf_df.to_excel(writer, sheet_name='LDFs_by_Loss_Year', index=False)

    # Sheet 4: Developed market ultimate by peril and year
    dev_summary.to_excel(writer, sheet_name='Developed_Ultimate', index=False)

    # Sheet 5: Emerging market ultimate by peril and year
    em_summary.to_excel(writer, sheet_name='Emerging_Ultimate', index=False)

    # Sheet 6: Inflation and frequency trends
    trends_df = pd.DataFrame([
        {'Line': l, 'Peril': p, 'Freq_Trend_Pct': freq_trend_by_peril.get((l, p), 0) * 100,
         'Inflation_Pct': inflation_by_line.get(l, 0) * 100}
        for l in ['motor', 'buildings', 'contents']
        for p in sorted(dev_claims[dev_claims['line'] == l]['peril'].unique())
    ])
    trends_df.to_excel(writer, sheet_name='Trends', index=False)

print(f'Saved: {excel_path}')
print(f'Sheets: Methodology, External_LDFs, LDFs_by_Loss_Year, Developed_Ultimate, Emerging_Ultimate, Trends')

## Caveats and Limitations

1. **External LDFs are industry averages, not company-specific.** The BI and subsidence LDFs from theactuarialclub.com are general benchmarks. Actual development may differ by portfolio mix, geography, and claims handling practices.

2. **2021 is not truly at ultimate for BI.** The external LDF at 42 months is 1.207x, meaning 2021 BI claims are only ~83% developed. The true ultimate may be higher if these claims develop differently from the industry average.

3. **Inflation trends derived from short-tail perils only.** We use closed-claim severity changes for short-tail perils (which are fully developed) as a proxy for pure inflation. This may not perfectly represent inflation for long-tail perils.

4. **Frequency trends from 2021-2023 only.** We avoid 2024 because IBNR distorts frequency. Only 3 years of trend data means trends may not be stable.

5. **Single data snapshot.** We cannot build true amount triangles (which require multiple snapshots of the same claims over time). The external LDFs substitute for this gap.

6. **Fire volatility.** Fire is high-severity/low-frequency — individual large claims dominate. The 2021 benchmark may not be representative. Emerging market fire 2023 (283 LR) was an outlier.

7. **Emerging market structural differences.** Emerging ultimate LRs are 40-90% of developed for most perils. Some of this gap is genuine (lower sum insured, different repair costs), not just underdevelopment.

8. **Part 1 comparison.** Part 1 used count-only factors for ALL perils (BI=1.82, subsidence=1.47). The new approach replaces these with external LDFs for BI and subsidence only. For short-tail perils, Part 1 factors are retained (they were count-based and appropriate since severity is fully developed).

## Files Saved
- Notebook: `notebooks_emerging_v2/04_BF_Development_Analysis_with_External_LDFs`
- Excel: `Excel files_emerging_v2/BF_Development_with_External_LDFs.xlsx`
- Triangles: `Excel files_emerging_v2/BF_Development_Triangles.xlsx`
- v3 proc files: `files_emerging_v2/proc_{motor,buildings,contents}_v3.parquet`
- v3 results: `files_emerging_v2/v3_results.pkl`
- Submissions: `submissions_emerging_v2/blend_v3_external_LDFs.csv` (primary, re-trained, Kaggle 0.12028), `submissions_emerging_v2/blend_calibrated_v2_external_LDFs.csv` (backup, scaled v2)
- No existing v1/v2 notebooks modified.
- All committed to GitHub `hackathon-bridget` branch.

## AI Assistance and Decision Log

### Chronology

| Date | Action | AI / Human | Rationale |
|------|--------|-----------|------------|
| Oct 25 | Part 1 BF factors audited — found count-only (no severity development) | AI | Traced 1.82 BI factor to count triangle: 2,146 mature count / 1,178 new count. No severity development applied. |
| Oct 25 | User supplied external LDFs from theactuarialclub.com (DOCX) | Human | User provided industry LDFs for BI and subsidence to replace count-only factors. |
| Oct 25 | Parsed DOCX, extracted mid-point cumulative incurred LDFs by development age | AI | Mid-points used where ranges provided. Interpolated to average development age per loss year. |
| Oct 25 | Applied external LDFs to BI and subsidence; retained Part 1 factors for short-tail | Joint | Short-tail perils confirmed negligible severity development via open/closed analysis. |
| Oct 25 | Inflation trends derived from short-tail closed claims (2021-2023) | AI | Short-tail perils fully developed, so severity change = pure inflation. Motor 6.3%, Buildings 5.8%, Contents 2.8%/yr. |
| Oct 25 | Frequency trends from developed market 2021-2023 (avoiding 2024 IBNR) | AI | 2024 counts underreported due to IBNR; 2021-2023 trends more stable. |
| Oct 25 | 2025 projections: 2024 ultimate x (1+freq trend) x (1+inflation) | Joint | User directed: do NOT apply trend to 2024 ultimate; trend only for 2025 predictions. |
| Oct 25 | Scaled v2 predictions by development ratio | AI | Faster than full retrain; preserves v2 model quality while reflecting external LDFs. Mean=328.55. |
| Oct 25 | Step 7: Updated proc files with external LDF targets (v3) | AI | Per-policy scaling: Motor +15%, Buildings +1.8%, Contents unchanged. |
| Oct 25 | Step 8: Re-trained XGBoost + GLM on v3 targets | AI | 5-fold CV. XGBoost v3 avg D2=0.048 (v2 0.060). GLM v3 avg D2=0.053 (v2 0.055). Blend v3 avg D2=0.064 (v2 0.067). |
| Oct 25 | GLM bug #1 fixed: in-place test modification across folds | AI | df_test was overwritten each fold. Fix: X_test_template copy per fold. |
| Oct 25 | GLM bug #2 fixed: sm.add_constant skipping constant | AI | Fix: has_constant='add'. |
| Oct 25 | Kaggle leaderboard: v3 scored 0.12028 | Human | Notable improvement over v2. Validates that lower CV D2 was a measurement artifact. |
| Oct 25 | Committed all files to GitHub hackathon-bridget branch | Joint | 10 files committed. |

### D2 Score Comparison

| Model | Motor | Buildings | Contents | Average |
|-------|-------|-----------|----------|---------|
| XGBoost v2 | 0.048 | 0.060 | 0.073 | 0.060 |
| XGBoost v3 | 0.035 | 0.028 | 0.081 | 0.048 |
| GLM v2 | 0.043 | 0.039 | 0.084 | 0.055 |
| GLM v3 (fixed) | 0.035 | 0.040 | 0.085 | 0.053 |
| Blend v2 | 0.049 | 0.070 | 0.083 | 0.067 |
| Blend v3 | 0.042 | 0.053 | 0.096 | 0.064 |

### Kaggle Leaderboard Result

**blend_v3_external_LDFs.csv scored 0.12028 on the Kaggle leaderboard** — a notable improvement over v2.

This validates the key insight: the lower CV D2 (0.064 vs v2 0.067) was a **measurement artifact of harder targets**, not worse predictions. The external LDF targets made the models learn higher BI/subsidence development patterns, which translated to better real-world predictions despite the training fit being worse.

### Key Decisions

1. **External LDFs for BI/subsidence supersede Part 1 count-only factors.** External LDFs capture both count and severity development; Part 1 only captured count. Justified by UK industry reference.

2. **Short-tail perils retain Part 1 count-based factors.** Open/closed analysis confirms severity development is negligible for own damage, theft, glass, EOW, flood, storm, accidental damage.

3. **2024 ultimate NOT adjusted by trend.** Trend factors (inflation x frequency) applied only to 2025 projections and emerging test predictions. Per user direction.

4. **v3 re-trained models are the primary submission (Kaggle score 0.12028).** Initially recommended scaled v2 as primary (higher CV D2), but Kaggle leaderboard confirmed v3 outperforms. The lower CV D2 was a measurement artifact of harder targets, not worse predictions. The models learned the external LDF development patterns and generalized better to the test set.

5. **Scaled v2 approach is now the backup.** blend_calibrated_v2_external_LDFs.csv (mean=328.55) remains available if needed.

6. **No per-claim/policy-level retraining with new ultimate_claims (deferred).** Full model retrain would require more time. Documented as next step.

### Files Produced

| File | Location | Purpose |
|------|----------|---------|
| 04_BF_Development_Analysis_with_External_LDFs.ipynb | notebooks_emerging_v2/ | Main analysis (Steps 1-8) |
| BF_Development_with_External_LDFs.xlsx | Excel files_emerging_v2/ | Methodology, LDFs, trends, summaries (6 sheets) |
| BF_Development_Triangles.xlsx | Excel files_emerging_v2/ | Count triangles, link ratios, open/closed (9 sheets) |
| proc_{line}_v3.parquet (x3) | files_emerging_v2/ | Updated proc files with external LDF targets |
| blend_v3_external_LDFs.csv | submissions_emerging_v2/ | **Primary submission** (re-trained v3, Kaggle 0.12028, mean=328.57) |
| blend_calibrated_v2_external_LDFs.csv | submissions_emerging_v2/ | Backup submission (scaled v2, mean=328.55) |
| v3_results.pkl | files_emerging_v2/ | Model D2 scores, blend weights, trend factors |

### External LDF Source
**URL:** https://theactuarialclub.com/2019/05/18/loss-development-factors-ldf/
**Type:** Cumulative age-to-ultimate incurred LDFs (industry averages, UK market)
**Use:** BI and subsidence only; replaces Part 1 count-only development factors for these long-tail perils